# 04 — Diseño Experimental: Harness

Este notebook define el motor que corre los experimentos:

- `run_experiment(model_factory, X, y, label_fractions, n_seeds, ...)` — ejecuta una grilla de (fracción × seed) y devuelve un DataFrame con métricas.
- `run_grid_search(model_factory, hyperparam_grid, X, y, label_fraction, n_seeds)` — para análisis de sensibilidad.

**Diseño:**
- Split 70/30 estratificado (justificación: dataset balanceado en órdenes de magnitud excepto BOMBAY; estratificar protege la clase minoritaria; 30% de test garantiza ~4000 puntos de evaluación).
- 5 seeds por configuración (justificación: balance entre coste computacional y estimación de varianza; con 5 seeds podemos reportar media ± std confiable).
- Lo que se controla: `random_state` del split y del modelo, `stratify=True`.
- Lo que se varía: `label_fraction ∈ {0.05, 0.10, 0.20}`, hiperparámetros (en grid search).
- Métricas: accuracy, F1-macro, recall-macro, tiempo de entrenamiento (s).

**Nota de integración:** este notebook NO importa modelos concretos. Recibe una `model_factory`
(callable que devuelve una instancia nueva del modelo). Esto desacopla el harness de la
implementación específica.

In [1]:
import numpy as np
import pandas as pd
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, recall_score
from tqdm import tqdm

# Importar funciones del data loader
%run 00_data_loader.ipynb

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = REPO_ROOT / 'results' / 'metrics'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Resultados se guardarán en: {RESULTS_DIR}")

Repo root: /home/escu/Documents/Universidad/Semestres/7moSemestre/mineriaDatos/Lab10-MD
Dataset: /home/escu/Documents/Universidad/Semestres/7moSemestre/mineriaDatos/Lab10-MD/data/processed/dataset_clean.csv
X.shape = (13543, 16)
y.shape = (13543,), clases únicas = [0 1 2 3 4 5 6]
features (16): ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4']
class_names: {0: 'BARBUNYA', 1: 'BOMBAY', 2: 'CALI', 3: 'DERMASON', 4: 'HOROZ', 5: 'SEKER', 6: 'SIRA'}
Etiquetas totales: 13543
Etiquetas visibles: 1355 (10.0%)
Etiquetas ocultas:  12188
Distribucion de clases visibles:
0    132
1     52
2    163
3    355
4    186
5    203
6    264
Name: count, dtype: int64
X_train: (9480, 16), X_test: (4063, 16)
y_train etiquetadas: 947 / 9480
y_test (todas etiquetadas): 4063
Resultados se guardarán en: /home/escu/Documents/Univ

In [2]:
def evaluate_model(y_true, y_pred):
    """Calcula todas las métricas estándar de clasificación multiclase."""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }

In [3]:
def run_experiment(
    model_factory,
    model_name,
    X,
    y,
    label_fractions=(0.05, 0.10, 0.20),
    n_seeds=5,
    test_size=0.30,
    verbose=True,
):
    """
    Corre el experimento principal: para cada combinación (label_fraction, seed),
    entrena el modelo y devuelve métricas.

    Parameters
    ----------
    model_factory : callable () -> estimator
        Función SIN argumentos que devuelve una instancia nueva del modelo.
        Ejemplo: `lambda: SemiSupervisedGMM(n_components=7, covariance_type='full')`.
    model_name : str
        Nombre para identificar el modelo en el CSV.
    X, y : arrays
        Dataset completo.
    label_fractions : tuple de floats
        Fracciones de etiquetas a probar.
    n_seeds : int
        Número de semillas por configuración.
    test_size : float
        Tamaño del test split.

    Returns
    -------
    pd.DataFrame con columnas:
        model, label_fraction, seed, accuracy, f1_macro, recall_macro, train_time_s
    """
    rows = []
    total = len(label_fractions) * n_seeds
    pbar = tqdm(total=total, desc=model_name, disable=not verbose)

    for lf in label_fractions:
        for seed in range(n_seeds):
            X_tr, X_te, y_tr_p, y_tr_full, y_te = make_semi_supervised_split(
                X, y, label_fraction=lf, test_size=test_size, random_state=seed
            )

            model = model_factory()

            t0 = time.perf_counter()
            try:
                model.fit(X_tr, y_tr_p)
                y_pred = model.predict(X_te)
                train_time = time.perf_counter() - t0
                metrics = evaluate_model(y_te, y_pred)
                error = None
            except Exception as e:
                train_time = time.perf_counter() - t0
                metrics = {'accuracy': np.nan, 'f1_macro': np.nan, 'recall_macro': np.nan}
                error = str(e)

            rows.append({
                'model': model_name,
                'label_fraction': lf,
                'seed': seed,
                **metrics,
                'train_time_s': train_time,
                'error': error,
            })
            pbar.update(1)

    pbar.close()
    return pd.DataFrame(rows)

In [4]:
from itertools import product

def run_grid_search(
    model_class,
    hyperparam_grid,
    X,
    y,
    label_fraction=0.10,
    n_seeds=3,
    test_size=0.30,
    verbose=True,
):
    """
    Análisis de sensibilidad: barre una grilla de hiperparámetros a una
    fracción de etiquetas fija y devuelve métricas por configuración.

    Parameters
    ----------
    model_class : clase del modelo (no instancia).
    hyperparam_grid : dict[str, list]
        Ejemplo: {'covariance_type': ['full', 'tied'], 'reg_covar': [1e-6, 1e-2]}
    label_fraction : float
        Fracción fija de etiquetas (default 10%).
    n_seeds : int
        Semillas por configuración (default 3, más bajo que el experimento principal).

    Returns
    -------
    pd.DataFrame con una fila por (configuración × seed).
    """
    keys = list(hyperparam_grid.keys())
    combos = list(product(*[hyperparam_grid[k] for k in keys]))
    rows = []

    pbar = tqdm(total=len(combos) * n_seeds, desc='grid', disable=not verbose)

    for combo in combos:
        params = dict(zip(keys, combo))
        for seed in range(n_seeds):
            X_tr, X_te, y_tr_p, y_tr_full, y_te = make_semi_supervised_split(
                X, y, label_fraction=label_fraction, test_size=test_size, random_state=seed
            )
            try:
                model = model_class(**params)
                t0 = time.perf_counter()
                model.fit(X_tr, y_tr_p)
                y_pred = model.predict(X_te)
                train_time = time.perf_counter() - t0
                metrics = evaluate_model(y_te, y_pred)
                error = None
            except Exception as e:
                metrics = {'accuracy': np.nan, 'f1_macro': np.nan, 'recall_macro': np.nan}
                train_time = np.nan
                error = str(e)

            rows.append({
                **params,
                'seed': seed,
                **metrics,
                'train_time_s': train_time,
                'error': error,
            })
            pbar.update(1)

    pbar.close()
    return pd.DataFrame(rows)

## Smoke test del harness

Para verificar que el harness funciona, lo probamos con dos modelos triviales de sklearn
(no son los del laboratorio, solo placeholders para validar el pipeline).

Cuando estén disponibles los modelos reales, este smoke test se reemplaza por una corrida real
en el notebook 05.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture

# Wrapper trivial: baseline supervisado que ignora -1
class _PlaceholderBaseline:
    """Placeholder para smoke test. Será reemplazado por los modelos del notebook 03."""
    def __init__(self):
        self.clf = LogisticRegression(max_iter=500, random_state=0)
    def fit(self, X, y_partial):
        mask = y_partial != -1
        self.clf.fit(X[mask], y_partial[mask])
        return self
    def predict(self, X):
        return self.clf.predict(X)

# Wrapper trivial: GMM puro (no semi-sup, solo unsupervised) con permutación greedy
class _PlaceholderGMM:
    """Placeholder. GMM sin uso de etiquetas + asignación de cluster->clase con las labeled."""
    def __init__(self, n_components=7, random_state=0):
        self.n_components = n_components
        self.gmm = GaussianMixture(n_components=n_components, random_state=random_state,
                                    covariance_type='full', reg_covar=1e-4, max_iter=200)
        self.cluster_to_class_ = None
    def fit(self, X, y_partial):
        self.gmm.fit(X)
        clusters = self.gmm.predict(X)
        mask = y_partial != -1
        # Mapear cada cluster a la clase mayoritaria entre las muestras etiquetadas de ese cluster
        mapping = {}
        for c in range(self.n_components):
            in_c = (clusters == c) & mask
            if in_c.sum() > 0:
                mapping[c] = int(pd.Series(y_partial[in_c]).mode().iloc[0])
            else:
                mapping[c] = 0  # fallback
        self.cluster_to_class_ = mapping
        return self
    def predict(self, X):
        clusters = self.gmm.predict(X)
        return np.array([self.cluster_to_class_[c] for c in clusters])

# Correr smoke test (rapido: 2 fracciones x 2 seeds)
df_smoke_base = run_experiment(
    model_factory=lambda: _PlaceholderBaseline(),
    model_name='placeholder_baseline',
    X=X, y=y,
    label_fractions=(0.05, 0.10),
    n_seeds=2,
)
df_smoke_gmm = run_experiment(
    model_factory=lambda: _PlaceholderGMM(n_components=7),
    model_name='placeholder_gmm',
    X=X, y=y,
    label_fractions=(0.05, 0.10),
    n_seeds=2,
)

df_smoke = pd.concat([df_smoke_base, df_smoke_gmm], ignore_index=True)
df_smoke

placeholder_gmm: 100%|██████████| 4/4 [00:01<00:00,  2.73it/s]


,model,label_fraction,seed,accuracy,f1_macro,recall_macro,train_time_s,error
0,placeholder_baseline,0.05,0,0.910411,0.921788,0.918590,0.007367,None
1,placeholder_baseline,0.05,1,0.923702,0.935378,0.933874,0.004822,None
2,placeholder_baseline,0.10,0,0.923209,0.936763,0.936308,0.008245,None
3,placeholder_baseline,0.10,1,0.920994,0.931287,0.930161,0.008910,None
4,placeholder_gmm,0.05,0,0.798917,0.740066,0.788573,0.516298,None
5,placeholder_gmm,0.05,1,0.804086,0.758800,0.788164,0.283740,None
6,placeholder_gmm,0.10,0,0.798917,0.740066,0.788573,0.330010,None
7,placeholder_gmm,0.10,1,0.810239,0.746391,0.792904,0.290365,None


In [6]:
smoke_path = RESULTS_DIR / 'metrics_smoke.csv'
df_smoke.to_csv(smoke_path, index=False)
print(f"Guardado: {smoke_path}")

# Resumen agregado (lo que tipicamente reportamos: media +/- std por modelo y fraccion)
summary = (
    df_smoke
    .groupby(['model', 'label_fraction'])
    .agg(
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        f1_mean=('f1_macro', 'mean'),
        f1_std=('f1_macro', 'std'),
        time_mean=('train_time_s', 'mean'),
    )
    .round(4)
)
summary

Guardado: /home/escu/Documents/Universidad/Semestres/7moSemestre/mineriaDatos/Lab10-MD/results/metrics/metrics_smoke.csv


accuracy_mean  accuracy_std  f1_mean  \
model                label_fraction                                         
placeholder_baseline 0.05                   0.9171        0.0094   0.9286   
                     0.10                   0.9221        0.0016   0.9340   
placeholder_gmm      0.05                   0.8015        0.0037   0.7494   
                     0.10                   0.8046        0.0080   0.7432   

                                     f1_std  time_mean  
model                label_fraction                     
placeholder_baseline 0.05            0.0096     0.0061  
                     0.10            0.0039     0.0086  
placeholder_gmm      0.05            0.0132     0.4000  
                     0.10            0.0045     0.3102

## Grillas de hiperparámetros sugeridas para el análisis de sensibilidad

**GMM semi-supervisado** (a fracción de 10%):
```python
GMM_GRID = {
    'covariance_type': ['full', 'tied', 'diag', 'spherical'],
    'reg_covar': [1e-6, 1e-4, 1e-2],
    'n_init': [1, 3],
}  # 4 x 3 x 2 = 24 combinaciones x 3 seeds = 72 corridas
```

**Constrained K-means** (a fracción de 10%):
```python
CKM_GRID = {
    'max_constraints': [50, 200, 500, 1000],
    'max_iter': [100, 300],
}  # 4 x 2 = 8 combinaciones x 3 seeds = 24 corridas
```

Para correrlos:
```python
df_sens_gmm = run_grid_search(SemiSupervisedGMM, GMM_GRID, X, y, label_fraction=0.10)
df_sens_gmm.to_csv(RESULTS_DIR / 'sensitivity_gmm.csv', index=False)
```